# exp04b — Control Positions

**Question:** Is the attention elevation found in exp04 specific to acrostic (sentence-initial)
positions, or is it a general property of stego CoT text?

**Design:** For each pair, compute attention curves for two sets of K positions:
- **Acrostic positions:** first token of each paragraph (the encoded letter)
- **Control positions:** token at offset +8 inside the same paragraph (mid-sentence)

Both sets are extracted from a **single forward pass** per text.

**Expected outcomes:**
- Elevation specific to acrostic positions → stego > open for acrostic only; control shows no gap
- Elevation general → stego > open for both acrostic and control

**Data:** `exp01/valid_pairs.json` (high-fidelity subset, same as exp04)

In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))

    if not os.path.exists('stego_CoT'):
        os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
    os.chdir('stego_CoT')
    os.system('pip install -q transformers accelerate scipy')

MODEL_ID    = 'meta-llama/Llama-3.1-8B-Instruct'
INPUT_DIR   = 'results/exp01'
OUTPUT_DIR  = 'results/exp04b'
N_MAX       = 80
CTRL_OFFSET = 8   # tokens past acrostic position → mid-sentence control

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('cwd:', os.getcwd())

In [ ]:
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer

with open(f'{INPUT_DIR}/valid_pairs.json') as f:
    all_pairs = json.load(f)

hi_fid = [p for p in all_pairs if p['fidelity'] == 1.0]
print(f'Total: {len(all_pairs)}, high-fidelity: {len(hi_fid)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',
)
model.eval()
model.config.use_cache = False

n_layers = model.config.num_hidden_layers
n_heads  = model.config.num_attention_heads
print(f'Layers: {n_layers}, heads: {n_heads}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def find_sentence_starts(token_ids, plen):
    """Return absolute token positions of paragraph starts in the CoT."""
    cot_ids   = list(token_ids[plen:])
    full_text = tokenizer.decode(cot_ids, skip_special_tokens=True)

    char_starts = []
    i = 0
    while i < len(full_text) and full_text[i] == '\n':
        i += 1
    if i < len(full_text):
        char_starts.append(i)
    while i < len(full_text):
        if full_text[i] == '\n':
            j = i + 1
            while j < len(full_text) and full_text[j] == '\n':
                j += 1
            if j - i >= 2 and j < len(full_text):
                char_starts.append(j)
            i = max(i + 1, j)
        else:
            i += 1

    positions = []
    found     = 0
    prev_len  = 0
    for tok_idx in range(len(cot_ids)):
        curr_len = len(tokenizer.decode(cot_ids[:tok_idx + 1], skip_special_tokens=True))
        while found < len(char_starts) and prev_len <= char_starts[found] < curr_len:
            positions.append(plen + tok_idx)
            found += 1
        prev_len = curr_len
        if found >= len(char_starts):
            break

    return positions


def make_control_positions(sent_positions, seq_len, offset):
    """Shift each acrostic position by offset, capped at the next sentence start."""
    ctrl = []
    for i, p in enumerate(sent_positions):
        next_p = sent_positions[i + 1] if i + 1 < len(sent_positions) else seq_len
        ctrl.append(min(p + offset, next_p - 1))
    return ctrl

In [ ]:
@torch.no_grad()
def extract_attentions(token_ids):
    """Run one forward pass and return per-layer attention tensors on CPU."""
    ids_t   = torch.tensor([token_ids], dtype=torch.long).to(model.device)
    outputs = model(ids_t, output_attentions=True)
    attn    = [a[0].cpu().float() for a in outputs.attentions]
    del outputs, ids_t
    torch.cuda.empty_cache()
    return attn


def curves_from_attentions(attentions, sent_positions):
    """Compute mean attention from sent_positions[K] to sent_positions[:K], per layer."""
    n_sent = len(sent_positions)
    result = np.zeros((n_sent, n_layers))
    for k in range(1, n_sent):
        pos_k = sent_positions[k]
        prev  = sent_positions[:k]
        for l, attn in enumerate(attentions):
            result[k, l] = attn[:, pos_k, prev].mean().item()
    return result


# Sanity check
ex    = hi_fid[0]
plen  = len(ex['payload'])
s_pos = find_sentence_starts(ex['stego_ids'], ex['stego_plen'])[:plen]
s_ctrl = make_control_positions(s_pos, len(ex['stego_ids']), CTRL_OFFSET)
print(f'Payload: {ex["payload"]}')
print('Acrostic positions:', s_pos)
print('Control  positions:', s_ctrl)
print('Acrostic tokens:', [tokenizer.decode([ex['stego_ids'][p]]) for p in s_pos])
print('Control  tokens:', [tokenizer.decode([ex['stego_ids'][p]]) for p in s_ctrl])

In [ ]:
# One forward pass per text; extract acrostic and control curves simultaneously
stego_acr  = []
open_acr   = []
stego_ctrl = []
open_ctrl  = []
skipped    = 0

for pair in hi_fid[:N_MAX]:
    plen  = len(pair['payload'])
    s_pos = find_sentence_starts(pair['stego_ids'], pair['stego_plen'])[:plen]
    o_pos = find_sentence_starts(pair['open_ids'],  pair['open_plen'])[:plen]
    n_pos = min(len(s_pos), len(o_pos))

    if n_pos < 2:
        skipped += 1
        continue

    s_pos  = s_pos[:n_pos]
    o_pos  = o_pos[:n_pos]
    sc_pos = make_control_positions(s_pos, len(pair['stego_ids']), CTRL_OFFSET)
    oc_pos = make_control_positions(o_pos, len(pair['open_ids']),  CTRL_OFFSET)

    s_attn = extract_attentions(pair['stego_ids'])
    stego_acr.append(curves_from_attentions(s_attn, s_pos))
    stego_ctrl.append(curves_from_attentions(s_attn, sc_pos))
    del s_attn

    o_attn = extract_attentions(pair['open_ids'])
    open_acr.append(curves_from_attentions(o_attn, o_pos))
    open_ctrl.append(curves_from_attentions(o_attn, oc_pos))
    del o_attn

    n = len(stego_acr)
    if n % 10 == 0:
        print(f'Processed {n} pairs (skipped {skipped})')

print(f'\nDone: {len(stego_acr)} pairs, {skipped} skipped')

In [ ]:
max_k  = max(c.shape[0] for c in stego_acr)
k_vals = list(range(1, max_k))

def collect_by_k(curves, k):
    return np.array([c[k].mean() for c in curves if k < c.shape[0]])

sa_means = [collect_by_k(stego_acr,  k).mean() for k in k_vals]
oa_means = [collect_by_k(open_acr,   k).mean() for k in k_vals]
sc_means = [collect_by_k(stego_ctrl, k).mean() for k in k_vals]
oc_means = [collect_by_k(open_ctrl,  k).mean() for k in k_vals]

sa_sems = [collect_by_k(stego_acr,  k).std() / np.sqrt(len(stego_acr))  for k in k_vals]
oa_sems = [collect_by_k(open_acr,   k).std() / np.sqrt(len(open_acr))   for k in k_vals]
sc_sems = [collect_by_k(stego_ctrl, k).std() / np.sqrt(len(stego_ctrl)) for k in k_vals]
oc_sems = [collect_by_k(open_ctrl,  k).std() / np.sqrt(len(open_ctrl))  for k in k_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)

for ax, title, s_m, s_e, o_m, o_e in [
    (axes[0], 'Acrostic positions (exp04 replication)', sa_means, sa_sems, oa_means, oa_sems),
    (axes[1], f'Control positions (+{CTRL_OFFSET} tokens, mid-sentence)', sc_means, sc_sems, oc_means, oc_sems),
]:
    ax.errorbar(k_vals, s_m, yerr=s_e, marker='o', label='stego', color='steelblue')
    ax.errorbar(k_vals, o_m, yerr=o_e, marker='s', label='open',  color='coral', linestyle='--')
    ax.set_xlabel('Sentence index K')
    ax.set_ylabel('Mean attention to positions {1..K-1}')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/control_vs_acrostic.png', dpi=150)
plt.show()

In [ ]:
def elevation_test(s_curves, o_curves, k_vals):
    for k in k_vals:
        diffs = [
            s_curves[i][k].mean() - o_curves[i][k].mean()
            for i in range(len(s_curves))
            if k < s_curves[i].shape[0] and k < o_curves[i].shape[0]
        ]
        if len(diffs) < 2:
            continue
        t, p = stats.ttest_1samp(diffs, 0)
        sig  = '  **' if p < 0.05 and np.mean(diffs) > 0 else ''
        print(f'  K={k}: mean diff = {np.mean(diffs):.5f}, t = {t:.3f}, p = {p:.4f}{sig}')


print('=== Elevation test: stego > open? ===')
print('Acrostic positions:')
elevation_test(stego_acr, open_acr, k_vals)
print('Control positions (mid-sentence):')
elevation_test(stego_ctrl, open_ctrl, k_vals)

# Direct comparison: acrostic elevation vs control elevation at each K
print('\n=== Acrostic vs control elevation (paired, stego only) ===')
for k in k_vals:
    acr_vals  = [stego_acr[i][k].mean()  for i in range(len(stego_acr))  if k < stego_acr[i].shape[0]]
    ctrl_vals = [stego_ctrl[i][k].mean() for i in range(len(stego_ctrl)) if k < stego_ctrl[i].shape[0]]
    n = min(len(acr_vals), len(ctrl_vals))
    if n < 2:
        continue
    diffs = np.array(acr_vals[:n]) - np.array(ctrl_vals[:n])
    t, p  = stats.ttest_1samp(diffs, 0)
    sig   = '  **' if p < 0.05 and diffs.mean() > 0 else ''
    print(f'  K={k}: acrostic - control = {diffs.mean():.5f}, t = {t:.3f}, p = {p:.4f}{sig}')

In [ ]:
summary = {
    'model':          MODEL_ID,
    'n_pairs':        len(stego_acr),
    'n_skipped':      skipped,
    'ctrl_offset':    CTRL_OFFSET,
    'k_vals':         k_vals,
    'acrostic_stego': [round(float(v), 6) for v in sa_means],
    'acrostic_open':  [round(float(v), 6) for v in oa_means],
    'control_stego':  [round(float(v), 6) for v in sc_means],
    'control_open':   [round(float(v), 6) for v in oc_means],
}

out_path = f'{OUTPUT_DIR}/exp04b_summary.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved:', out_path)

if IN_COLAB:
    from google.colab import drive
    import shutil, pathlib
    drive.mount('/content/drive')
    dst = pathlib.Path('/content/drive/MyDrive/stego_cot_results/exp04b')
    dst.mkdir(parents=True, exist_ok=True)
    for fp in pathlib.Path(OUTPUT_DIR).iterdir():
        shutil.copy(fp, dst / fp.name)
        print(f'  -> Drive: {fp.name}')